In [1]:
import base64
import os
import requests
import json
from dotenv import load_dotenv

load_dotenv()

API_URL = os.getenv("PaddleOCR_VL_API_URL")
TOKEN = os.getenv("PaddleOCR_VL_API_TOKEN")


In [2]:
def process_document_with_paddle(file_path, api_url, token, output_dir="output", **options):
    """
    Sends a file to the PaddleOCR-VL API and saves the resulting Markdown and images.
    
    Args:
        file_path (str): Path to the input image or PDF.
        api_url (str): The endpoint URL for the OCR service.
        token (str): Your API authorization token.
        output_dir (str): Directory where results will be saved.
        **options: Optional flags like useDocUnwarping, useChartRecognition, etc.
    """

    doc_name = os.path.splitext(os.path.basename(file_path))[0]
    output_dir = os.path.join(output_dir, doc_name)
    
    file_extension = os.path.splitext(file_path)[1].lower()
    file_type = 0 if file_extension == ".pdf" else 1
    
    try:
        with open(file_path, "rb") as file:
            file_data = base64.b64encode(file.read()).decode("ascii")
    except FileNotFoundError:
        print(f"Error: File {file_path} not found.")
        return

    headers = {
        "Authorization": f"token {token}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "file": file_data,
        "fileType": file_type,
        "useDocOrientationClassify": options.get("useDocOrientationClassify", False),
        "useDocUnwarping": options.get("useDocUnwarping", False),
        "useChartRecognition": options.get("useChartRecognition", False),
    }

    print(f"Processing {file_path}...")
    response = requests.post(api_url, json=payload, headers=headers)
    
    if response.status_code != 200:
        print(f"API Error: {response.status_code} - {response.text}")
        return

    result = response.json().get("result", {})
    os.makedirs(output_dir, exist_ok=True)

    # Parse and Save Results
    for i, res in enumerate(result.get("layoutParsingResults", [])):
        # Save Markdown
        md_text = res.get("markdown", {}).get("text", "")
        md_filename = os.path.join(output_dir, f"doc_{i}.md")
        with open(md_filename, "w", encoding="utf-8") as md_file:
            md_file.write(md_text)
        
        # Save Inline Markdown Images
        for img_path, img_url in res.get("markdown", {}).get("images", {}).items():
            _save_image_from_url(img_url, os.path.join(output_dir, img_path))
            
        # Save Layout/Output Images
        for img_name, img_url in res.get("outputImages", {}).items():
            save_path = os.path.join(output_dir, f"{img_name}_{i}.jpg")
            _save_image_from_url(img_url, save_path)

        # Save Pruned Result
        pruned_result = res.get("prunedResult", {})
        with open(os.path.join(output_dir, f"pruned_result_{i}.json"), "w", encoding="utf-8") as json_file:
            json.dump(pruned_result, json_file, indent=4)

    print(f"Processing complete. Files saved in: {output_dir}")

def _save_image_from_url(url, save_path):
    """Helper to download and save images."""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    img_response = requests.get(url)
    if img_response.status_code == 200:
        with open(save_path, "wb") as f:
            f.write(img_response.content)
    else:
        print(f"Failed to download image from {url}")

In [ ]:
# 37s (1st Trial), 24.6s (2nd Trial) for processing
process_document_with_paddle(
    file_path="../data/form_test_1.png",
    api_url=API_URL,
    token=TOKEN,
    output_dir="output"
)

Processing ../data/form_test_1.png...
Processing complete. Files saved in: output\form_test_1


In [ ]:
# 34.2s (1st Trial), 59.8s (2nd Trial) for processing
process_document_with_paddle(
    file_path="../data/handwritten_text_test_1.png",
    api_url=API_URL,
    token=TOKEN,
    output_dir="output"
)

Processing ../data/handwritten_text_test_1.png...
Processing complete. Files saved in: output\handwritten_text_test_1


In [ ]:
# 84.6s (1st Trial), 
process_document_with_paddle(
    file_path="../data/utar_form_digital.pdf",
    api_url=API_URL,
    token=TOKEN,
    output_dir="output"
)

Processing ../data/utar_form_digital.pdf...
Processing complete. Files saved in: output\utar_form_digital


## References
PaddleCOR API Key can get from: https://aistudio.baidu.com/paddleocr